# nb12 -- Pathway / complex / regulatory validation
Checks whether `lasso`'s non-cognate top-predictor genes are known biological
partners of the cognate gene, using three complementary sources:

1. **STRING** -- functional/physical association, broad coverage. Pulls the
   per-evidence-channel scores (experiments, curated databases, co-expression,
   text-mining), not just the combined score, so a hit backed by hard evidence
   can be told apart from one that's mostly literature text-mining.
2. **OmniPath complexes** -- `CORUM, ComplexPortal, hu.MAP`.
3. **TRRUST** -- literature-curated transcription-factor -> target-gene
   regulatory interactions, checked in both directions.

**Model:** `lasso` -- the variant carried forward from nb11 (best cognate
rank-1, 81.6% bootstrap-stable, most reproducible across every stress test run).

**Scope:** the gray-zone proteins -- neither in the trustworthy core (cognate
rank-1 AND bootstrap-stable) nor flagged as likely artifacts (weak raw
correlation / technical gene / large rank gap despite strong raw correlation).
These are the ambiguous cases: the model's top pick isn't the literal cognate
gene, but might still be a real biological partner rather than noise.

## Environment setup

In [1]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/covid_citeseq_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

Mounted at /content/drive
Running on Colab | BASE_PATH = /content/drive/MyDrive/covid_citeseq_project


## Imports and config

In [2]:
import sys
sys.path.insert(0, str(BASE_PATH))

import pandas as pd

from src.analysis.pathway_validation import (
    build_trustworthy_core, flag_artifacts, build_query_set,
    fetch_string_partner_cache, string_validate_pairs,
    load_omnipath_complexes, build_gene_to_complexes, complex_validate_pairs,
    load_trrust, trrust_validate_pairs,
    combined_verdict,
    OMNIPATH_COMPLEX_RESOURCES,
)

MODEL_VARIANT = 'lasso'

NB11_RESULTS_DIR = BASE_PATH / 'results' / 'mlp_variant_comparison'
RESULTS_DIR = BASE_PATH / 'results' / 'pathway_validation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MIN_BOOTSTRAP_FREQUENCY = 0.8

## Load nb11 outputs and build the query set
`cognate_ranks`, `bootstrap_rank1`, and `raw_correlation_check` are all
already saved from nb11's run for `lasso` -- no retraining needed here.

In [3]:
cognate_ranks = pd.read_csv(NB11_RESULTS_DIR / f'nb11_{MODEL_VARIANT}_cognate_ranks.csv')
bootstrap_rank1 = pd.read_csv(NB11_RESULTS_DIR / f'nb11_{MODEL_VARIANT}_bootstrap_rank1.csv')
raw_corr_check = pd.read_csv(NB11_RESULTS_DIR / f'nb11_{MODEL_VARIANT}_raw_correlation_check.csv')

print(f'cognate_ranks:   {len(cognate_ranks)} proteins')
print(f'bootstrap_rank1: {len(bootstrap_rank1)} proteins')
print(f'raw_corr_check:  {len(raw_corr_check)} proteins')

cognate_ranks:   163 proteins
bootstrap_rank1: 163 proteins
raw_corr_check:  163 proteins


In [4]:
cognate_ranks

,protein,cognate_gene,cognate_rank,n_genes,top_predictor_gene
0,AB_CD123,IL3RA,1,2092,IL3RA
1,AB_CD34,CD34,1,2092,CD34
2,AB_CCR4,CCR4,1,2092,CCR4
3,AB_CD7,CD7,1,2092,CD7
4,AB_ITGAX,ITGAX,1,2092,ITGAX
...,...,...,...,...,...
158,AB_TNFRSF8,TNFRSF8,1948,2092,MTRNR2L8
159,AB_THY1,THY1,1962,2092,EEF1A1
160,AB_TSLPR,CRLF2,2030,2092,HBB
161,AB_CD1a,CD1A,2042,2092,HBB


In [4]:
trustworthy_core = build_trustworthy_core(cognate_ranks, bootstrap_rank1, MIN_BOOTSTRAP_FREQUENCY)
artifact_flags = flag_artifacts(raw_corr_check)
query_set = build_query_set(cognate_ranks, trustworthy_core, artifact_flags)

trustworthy_proteins = set(trustworthy_core.loc[trustworthy_core['trustworthy'], 'protein'])
artifact_proteins = set(artifact_flags.loc[artifact_flags['likely_artifact'], 'protein'])

print(f'Trustworthy core (skipped): {len(trustworthy_proteins)}')
print(f'Artifact-flagged (excluded): {len(artifact_proteins)}')
print(f'Query set: {len(query_set)}')

trustworthy_core.to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_trustworthy_core.csv', index=False)
artifact_flags.to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_artifact_flags.csv', index=False)
query_set

Trustworthy core (skipped): 36
Artifact-flagged (excluded): 67
Query set: 58


,protein,cognate_gene,cognate_rank,n_genes,top_predictor_gene
38,AB_CD8,CD8A,2,2092,CD8B
39,AB_CD14,CD14,2,2092,CXCL2
40,AB_DPP4,DPP4,2,2092,SLC4A10
43,AB_CD27,CD27,2,2092,EEF1A1
44,AB_CD25,IL2RA,2,2092,FOXP3
45,AB_KLRD1,KLRD1,3,2092,KLRC1
47,AB_CD5,CD5,3,2092,EEF1A1
48,AB_CD19,CD19,4,2092,CD79A
49,AB_CD52,CD52,4,2092,EEF1A1
50,AB_CXCR5,CXCR5,4,2092,FCER2


## 1. STRING check -- combined score AND per-evidence-channel breakdown
`escore` (experimental) and `dscore` (curated databases) are hard-evidence
channels; `tscore` (text-mining) is the weakest, most inflatable one.

In [5]:
unique_query_genes = pd.unique(query_set[['cognate_gene', 'top_predictor_gene']].values.ravel())
print(f'Fetching STRING partners for {len(unique_query_genes)} unique genes...')

string_partner_cache = fetch_string_partner_cache(unique_query_genes)
print('STRING fetch complete.')

string_results = string_validate_pairs(query_set, string_partner_cache)

print(f"STRING-supported pairs: {string_results['string_known_interaction'].sum()} / {len(string_results)}")
print(f"  ...with hard evidence (experiments or curated database): {string_results['string_hard_evidence'].sum()}")
print(f"  ...text-mining evidence only: {string_results['string_textmining_only'].sum()}")
string_results[string_results['string_known_interaction']].sort_values('string_score', ascending=False)

Fetching STRING partners for 80 unique genes...
  10/80...
  20/80...
  30/80...
  40/80...
  50/80...
  60/80...
  70/80...
  80/80...
STRING fetch complete.
STRING-supported pairs: 20 / 58
  ...with hard evidence (experiments or curated database): 8
  ...text-mining evidence only: 9


,protein,cognate_gene,cognate_rank,n_genes,top_predictor_gene,string_known_interaction,string_score,string_escore,string_dscore,string_tscore,string_ascore,string_nscore,string_fscore,string_pscore,string_high_confidence,string_hard_evidence,string_textmining_only
0,AB_CD8,CD8A,2,2092,CD8B,True,0.999,0.691,0.9,0.892,0.819,0.0,0.0,0.000,True,True,False
5,AB_KLRD1,KLRD1,3,2092,KLRC1,True,0.999,0.982,0.9,0.986,0.096,0.0,0.0,0.000,True,True,False
7,AB_CD19,CD19,4,2092,CD79A,True,0.998,0.292,0.5,0.970,0.829,0.0,0.0,0.000,True,True,False
17,AB_CD4,CD4,6,2092,CD8B,True,0.980,0.000,0.9,0.785,0.166,0.0,0.0,0.000,True,True,False
4,AB_CD25,IL2RA,2,2092,FOXP3,True,0.970,0.288,0.5,0.916,0.142,0.0,0.0,0.000,True,True,False
14,AB_CD22,CD22,5,2092,MS4A1,True,0.928,0.092,0.0,0.807,0.542,0.0,0.0,0.215,True,False,True
34,AB_ITGB7,ITGB7,37,2092,ITGB1,True,0.913,0.000,0.9,0.139,0.000,0.0,0.0,0.075,True,True,False
39,AB_FCGR2A,FCGR2A,219,2092,FCGR2B,True,0.892,0.471,0.0,0.706,0.356,0.0,0.0,0.053,True,True,False
47,AB_SELP,SELP,971,2092,PPBP,True,0.887,0.000,0.0,0.862,0.219,0.0,0.0,0.000,True,False,True
23,AB_CD21,CR2,13,2092,CD79A,True,0.886,0.000,0.0,0.829,0.363,0.0,0.0,0.000,True,False,True


## 2. OmniPath complexes
`CORUM, ComplexPortal, hu.MAP` -- CORUM alone tends to have narrow curation
scope; hu.MAP adds AP-MS-based complexes rather than literature curation alone.

In [6]:
complex_cache_path = RESULTS_DIR / f'omnipath_complexes_{OMNIPATH_COMPLEX_RESOURCES.replace(",", "_")}.tsv'
complexes = load_omnipath_complexes(OMNIPATH_COMPLEX_RESOURCES, cache_path=complex_cache_path)
print(f'Complexes loaded ({OMNIPATH_COMPLEX_RESOURCES}): {len(complexes)}')

gene_to_complexes = build_gene_to_complexes(complexes)
print(f'Genes indexed: {len(gene_to_complexes)}')

complex_results = complex_validate_pairs(query_set, gene_to_complexes)
print(f"Complex-supported pairs: {complex_results['shared_complex'].sum()} / {len(complex_results)}")
complex_results[complex_results['shared_complex']]

Complexes loaded (CORUM,ComplexPortal,hu.MAP): 2734
Genes indexed: 3672
Complex-supported pairs: 0 / 58


,protein,cognate_gene,cognate_rank,n_genes,top_predictor_gene,shared_complex,complex_names


## 3. TRRUST -- transcription factor -> target regulatory check
Checks whether either gene in a pair is a known TF that regulates the other,
in either direction. If the download fails, get `trrust_rawdata.human.tsv`
manually from https://www.grnpedia.org/trrust/ and place it at
`RESULTS_DIR / 'trrust_rawdata.human.tsv'`.

In [7]:
trrust_cache_path = RESULTS_DIR / 'trrust_rawdata.human.tsv'
trrust = load_trrust(cache_path=trrust_cache_path)
print(f'TRRUST interactions loaded: {len(trrust)}')

trrust_results = trrust_validate_pairs(query_set, trrust)
print(f"TRRUST-supported (regulatory) pairs: {trrust_results['trrust_regulatory_pair'].sum()} / {len(trrust_results)}")
trrust_results[trrust_results['trrust_regulatory_pair']]

TRRUST interactions loaded: 9398
TRRUST-supported (regulatory) pairs: 1 / 58


,protein,cognate_gene,cognate_rank,n_genes,top_predictor_gene,trrust_regulatory_pair,trrust_direction,trrust_regtype
4,AB_CD25,IL2RA,2,2092,FOXP3,True,FOXP3->IL2RA,Activation


## Combined verdict
A pair counts as biologically supported if STRING, the complex check, or
TRRUST confirms it. `evidence_type` distinguishes physical (complex),
regulatory (TRRUST), and general-association (STRING) support, and separates
hard evidence from text-mining-only STRING hits.

In [ ]:
combined = combined_verdict(string_results, complex_results, trrust_results)
combined = combined.merge(query_set[['protein', 'cognate_rank']], on='protein')

print(f"Total queried: {len(combined)}")
print(f"Biologically supported (any source): {combined['biologically_supported'].sum()}")
print(f"Unexplained (no source supports): {(~combined['biologically_supported']).sum()}")
print('\nBy evidence type:')
print(combined['evidence_type'].value_counts())

combined.sort_values(['biologically_supported', 'string_score'], ascending=[False, False])

Total queried: 58
Biologically supported (any source): 20
Unexplained (no source supports): 38

By evidence type:
evidence_type
none                          38
string_textmining_or_other    12
string_hard_evidence           7
regulatory                     1
Name: count, dtype: int64


,protein,string_known_interaction,string_hard_evidence,string_score,shared_complex,complex_names,trrust_regulatory_pair,trrust_direction,trrust_regtype,biologically_supported,evidence_type,cognate_rank
0,AB_CD8,True,True,0.999,False,,False,None,None,True,string_hard_evidence,2
5,AB_KLRD1,True,True,0.999,False,,False,None,None,True,string_hard_evidence,3
7,AB_CD19,True,True,0.998,False,,False,None,None,True,string_hard_evidence,4
17,AB_CD4,True,True,0.980,False,,False,None,None,True,string_hard_evidence,6
4,AB_CD25,True,True,0.970,False,,True,FOXP3->IL2RA,Activation,True,regulatory,2
14,AB_CD22,True,False,0.928,False,,False,None,None,True,string_textmining_or_other,5
34,AB_ITGB7,True,True,0.913,False,,False,None,None,True,string_hard_evidence,37
39,AB_FCGR2A,True,True,0.892,False,,False,None,None,True,string_hard_evidence,219
47,AB_SELP,True,False,0.887,False,,False,None,None,True,string_textmining_or_other,971
23,AB_CD21,True,False,0.886,False,,False,None,None,True,string_textmining_or_other,13


## Save results

In [ ]:
combined.to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_pathway_validation_results.csv', index=False)
combined[combined['biologically_supported']].to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_biologically_supported_pairs.csv', index=False)
combined[~combined['biologically_supported']].to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_unexplained_pairs.csv', index=False)

print(f'Saved to {RESULTS_DIR}')
print(f'  {MODEL_VARIANT}_trustworthy_core.csv             -- cognate rank-1 AND bootstrap-stable, {trustworthy_core["trustworthy"].sum()}/{len(trustworthy_core)}')
print(f'  {MODEL_VARIANT}_artifact_flags.csv                -- likely-artifact top picks, {artifact_flags["likely_artifact"].sum()}/{len(artifact_flags)}')
print(f'  {MODEL_VARIANT}_pathway_validation_results.csv    -- STRING + complexes + TRRUST, all queried (gray-zone) pairs')
print(f'  {MODEL_VARIANT}_biologically_supported_pairs.csv  -- confirmed by at least one source, with evidence_type')
print(f'  {MODEL_VARIANT}_unexplained_pairs.csv             -- no source supports -- literature spot-check or exclusion candidates')

## Summary
Fill in after running:
- Trustworthy core size (cognate rank-1 AND bootstrap-stable):
- Artifact-flagged size:
- Query set (gray-zone) size:
- Biologically supported fraction of the gray zone:
- Notable regulatory (TRRUST) or complex hits worth highlighting in the pitch:
- Unexplained pairs worth a manual literature check: